# Monitoring Rules & Gold Alerts (`pipelines/04_monitoring_rules.ipynb`)

This notebook reads accepted Silver transactions (`workspace.sentinel_silver.transactions`), computes earlier activity features, evaluates three provisional monitoring rules, builds analyst evidence, and writes flagged alerts to `workspace.sentinel_gold.alerts`.

| Rule ID | Version | Window (`[lower, upper)`) | Match Condition | Example |
| :--- | :--- | :--- | :--- | :--- |
| `RULE_01_CARD_VELOCITY_1H` | `v1.0-provisional` | `[event_time - 1h, event_time)` | `card_txn_count_1h >= 3` | `3` earlier card payments in the last hour -> **Match** |
| `RULE_02_UNUSUAL_AMOUNT_14D` | `v1.0-provisional` | `[event_time - 14d, event_time)` | `card_txn_count_14d >= 5` and `amount >= 3.00 * card_avg_amount_14d` | `5` prior payments averaging `20.00` (`3x = 60.00`); `amount = 75.00` -> **Match** |
| `RULE_03_REPEATED_SIMILAR_MERCHANT_1H` | `v1.0-provisional` | `[event_time - 1h, event_time)` | `card_merchant_similar_amount_count_1h >= 2` (within `±10%` of `amount`) | `amount = 100.00` with earlier same-merchant payments of `90.00` and `110.00` in 1h -> **Match** |

In [ ]:
# Configuration: imports, table names, provisional thresholds, and Gold output columns
from decimal import Decimal
import os
import sys
from pyspark.sql import functions as F

for _candidate_dir in (os.getcwd(), os.path.join(os.getcwd(), "pipelines")):
    if os.path.isfile(os.path.join(_candidate_dir, "historical_features.py")) and _candidate_dir not in sys.path:
        sys.path.insert(0, _candidate_dir)

from historical_features import FEATURE_COLUMNS, compute_historical_features

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
GOLD_SCHEMA = "workspace.sentinel_gold"
GOLD_ALERTS_TABLE = f"{GOLD_SCHEMA}.alerts"
ACTIVE_RULESET_VERSION = "ruleset-v1.0-provisional"

# Rule 1: Velocity — at least 3 earlier card payments within 1 hour
RULE_01_ID = "RULE_01_CARD_VELOCITY_1H"
RULE_01_VERSION = "v1.0-provisional"
RULE_01_WINDOW = "[event_time - 1h, event_time)"
RULE_01_MIN_PRIOR_CARD_TXNS_1H = 3

# Rule 2: Unusual amount — amount >= 3x earlier 14-day average, with at least 5 earlier payments
RULE_02_ID = "RULE_02_UNUSUAL_AMOUNT_14D"
RULE_02_VERSION = "v1.0-provisional"
RULE_02_WINDOW = "[event_time - 14d, event_time)"
RULE_02_MIN_PRIOR_CARD_TXNS_14D = 5
RULE_02_AMOUNT_MULTIPLIER = Decimal("3.00")

# Rule 3: Repeated similar payments — at least 2 earlier same-merchant payments in 1h within +/-10% amount
RULE_03_ID = "RULE_03_REPEATED_SIMILAR_MERCHANT_1H"
RULE_03_VERSION = "v1.0-provisional"
RULE_03_WINDOW = "[event_time - 1h, event_time)"
RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H = 2
RULE_03_AMOUNT_TOLERANCE_PCT = Decimal("0.10")

GOLD_ALERTS_COLUMNS = [
    "alert_id",
    "trans_num",
    "ruleset_version",
    "event_time",
    "cc_num",
    "merchant",
    "category",
    "amount",
    "replay_batch",
    "source_file",
    "source_row_index",
    "card_txn_count_1h",
    "card_txn_count_14d",
    "card_avg_amount_14d",
    "unusual_amount_threshold_14d",
    "card_merchant_txn_count_1h",
    "card_merchant_similar_amount_count_1h",
    "similar_amount_lower_bound",
    "similar_amount_upper_bound",
    "matched_rule_count",
    "matched_rule_ids",
    "matched_rule_versions",
    "rule_evidence",
]

### Step 1: Read Silver Transactions & Compute Historical Features

We load `workspace.sentinel_silver.transactions`, confirm `is_fraud` is absent, and compute earlier-only features (`< event_time`, inclusive of `-1h` and `-14d`).
- **Example**: For a payment at `12:00:00` with `amount = 100.00`, the 1-hour window is `[11:00:00, 12:00:00)` and the `±10%` similarity band is `[90.00, 110.00]`.

In [ ]:
# Step 1: Read Silver transactions and compute historical features
silver_transactions = spark.read.table(SILVER_TRANSACTIONS_TABLE)
if "is_fraud" in silver_transactions.columns:
    raise AssertionError("Label leakage guard failed: 'is_fraud' must not exist in Silver transactions.")

silver_with_features = compute_historical_features(
    silver_transactions,
    similar_amount_tolerance_pct=RULE_03_AMOUNT_TOLERANCE_PCT,
)

feature_counts = silver_with_features.agg(
    F.count("*").alias("feature_rows"),
    F.countDistinct("trans_num").alias("distinct_trans_num"),
).collect()[0]
print(
    f"Computed features for {feature_counts['feature_rows']:,} transactions "
    f"({feature_counts['distinct_trans_num']:,} unique trans_num)."
)

### Step 2: Evaluate the Three Rule Flags

We add reference threshold columns (`unusual_amount_threshold_14d`, `similar_amount_lower_bound`, `similar_amount_upper_bound`) and boolean match columns (`rule_01_velocity_match`, `rule_02_unusual_amount_match`, `rule_03_similar_merchant_match`).
- **Example**: If `card_txn_count_14d = 4` and `card_avg_amount_14d = 20.00`, even `amount = 100.00` gives `rule_02_unusual_amount_match = False` because Rule 2 requires at least `5` prior payments.

In [ ]:
# Step 2: Calculate reference bounds and boolean match flags for Rules 1, 2, and 3
lower_factor = F.lit(Decimal("1.00") - RULE_03_AMOUNT_TOLERANCE_PCT)
upper_factor = F.lit(Decimal("1.00") + RULE_03_AMOUNT_TOLERANCE_PCT)

flagged_df = (
    silver_with_features.withColumn(
        "unusual_amount_threshold_14d",
        F.when(
            F.col("card_avg_amount_14d").isNotNull(),
            F.round(
                F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER), 2
            ).cast("decimal(18,2)"),
        ).otherwise(F.lit(None).cast("decimal(18,2)")),
    )
    .withColumn(
        "similar_amount_lower_bound",
        F.round(F.col("amount") * lower_factor, 2).cast("decimal(18,2)"),
    )
    .withColumn(
        "similar_amount_upper_bound",
        F.round(F.col("amount") * upper_factor, 2).cast("decimal(18,2)"),
    )
    .withColumn(
        "rule_01_velocity_match",
        F.col("card_txn_count_1h") >= F.lit(RULE_01_MIN_PRIOR_CARD_TXNS_1H),
    )
    .withColumn(
        "rule_02_unusual_amount_match",
        (F.col("card_txn_count_14d") >= F.lit(RULE_02_MIN_PRIOR_CARD_TXNS_14D))
        & F.col("card_avg_amount_14d").isNotNull()
        & (
            F.col("amount")
            >= (F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER))
        ),
    )
    .withColumn(
        "rule_03_similar_merchant_match",
        F.col("card_merchant_similar_amount_count_1h")
        >= F.lit(RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H),
    )
)

flag_counts = flagged_df.agg(
    F.sum(F.when(F.col("rule_01_velocity_match"), 1).otherwise(0)).alias("rule_01_matches"),
    F.sum(F.when(F.col("rule_02_unusual_amount_match"), 1).otherwise(0)).alias("rule_02_matches"),
    F.sum(F.when(F.col("rule_03_similar_merchant_match"), 1).otherwise(0)).alias("rule_03_matches"),
).collect()[0]
print(
    f"Rule matches — Rule 1 (velocity): {flag_counts['rule_01_matches']:,}, "
    f"Rule 2 (unusual amount): {flag_counts['rule_02_matches']:,}, "
    f"Rule 3 (similar merchant): {flag_counts['rule_03_matches']:,}"
)

### Step 3a: Build Rule 1 Evidence (`F.when` and `F.struct`)

- **`F.when(condition, value)`**: Works like an `if` statement per row. When `condition` is `True`, it returns `value`; without `.otherwise(...)`, rows where `condition` is `False` become `NULL`.
- **`F.struct(...)`**: Bundles several named columns into one mini-record (like a dictionary) inside a single column.
- **Tiny Example**: `F.when(F.col("rule_01_velocity_match"), F.struct(F.lit("R1").alias("rule_id")))` produces `{"rule_id": "R1"}` on a matching row and `NULL` on a non-matching row.

In [ ]:
# Step 3a: Build evidence struct for Rule 1 (Velocity) when matched, else NULL
rule_01_evidence = F.when(
    F.col("rule_01_velocity_match"),
    F.struct(
        F.lit(RULE_01_ID).alias("rule_id"),
        F.lit(RULE_01_VERSION).alias("rule_version"),
        F.lit(RULE_01_WINDOW).alias("window"),
        F.concat(
            F.lit("card_txn_count_1h="),
            F.col("card_txn_count_1h").cast("string"),
        ).alias("observed_metrics"),
        F.lit(f"card_txn_count_1h >= {RULE_01_MIN_PRIOR_CARD_TXNS_1H}").alias(
            "threshold_definition"
        ),
    ),
)

### Step 3b: Build Rule 2 Evidence (Unusual Amount)

Uses the same `F.when` + `F.struct` pattern to record the transaction `amount`, 14-day average, 14-day payment count, and `3.00x` threshold when `rule_02_unusual_amount_match` is `True`.
- **Tiny Example**: `observed_metrics = "amount=60.00, card_avg_amount_14d=20.00, card_txn_count_14d=5"`.

In [ ]:
# Step 3b: Build evidence struct for Rule 2 (Unusual Amount) when matched, else NULL
rule_02_evidence = F.when(
    F.col("rule_02_unusual_amount_match"),
    F.struct(
        F.lit(RULE_02_ID).alias("rule_id"),
        F.lit(RULE_02_VERSION).alias("rule_version"),
        F.lit(RULE_02_WINDOW).alias("window"),
        F.concat(
            F.lit("amount="),
            F.col("amount").cast("string"),
            F.lit(", card_avg_amount_14d="),
            F.col("card_avg_amount_14d").cast("string"),
            F.lit(", card_txn_count_14d="),
            F.col("card_txn_count_14d").cast("string"),
        ).alias("observed_metrics"),
        F.concat(
            F.lit(
                f"card_txn_count_14d >= {RULE_02_MIN_PRIOR_CARD_TXNS_14D} AND amount >= {RULE_02_AMOUNT_MULTIPLIER} * card_avg_amount_14d ("
            ),
            F.col("unusual_amount_threshold_14d").cast("string"),
            F.lit(")"),
        ).alias("threshold_definition"),
    ),
)

### Step 3c: Build Rule 3 Evidence (Repeated Similar Merchant Payments)

Records `amount`, the similar-amount merchant count (`card_merchant_similar_amount_count_1h`), total 1-hour merchant count (`card_merchant_txn_count_1h`), and the `[lower, upper]` `±10%` bounds when `rule_03_similar_merchant_match` is `True`.
- **Tiny Example**: For `amount = 100.00`, `threshold_definition` ends with `within +/-10% [90.00, 110.00]`.

In [ ]:
# Step 3c: Build evidence struct for Rule 3 (Repeated Similar Merchant) when matched, else NULL
rule_03_evidence = F.when(
    F.col("rule_03_similar_merchant_match"),
    F.struct(
        F.lit(RULE_03_ID).alias("rule_id"),
        F.lit(RULE_03_VERSION).alias("rule_version"),
        F.lit(RULE_03_WINDOW).alias("window"),
        F.concat(
            F.lit("amount="),
            F.col("amount").cast("string"),
            F.lit(", card_merchant_similar_amount_count_1h="),
            F.col("card_merchant_similar_amount_count_1h").cast("string"),
            F.lit(", card_merchant_txn_count_1h="),
            F.col("card_merchant_txn_count_1h").cast("string"),
        ).alias("observed_metrics"),
        F.concat(
            F.lit(
                f"card_merchant_similar_amount_count_1h >= {RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H} within +/-10% ["
            ),
            F.col("similar_amount_lower_bound").cast("string"),
            F.lit(", "),
            F.col("similar_amount_upper_bound").cast("string"),
            F.lit("]"),
        ).alias("threshold_definition"),
    ),
)

### Step 3d: Combine Evidence per Transaction (`F.array`, `F.filter`, and `F.transform`)

- **`F.array(c1, c2, c3)`**: Collects the three rule evidence expressions into one list per row. Unmatched rules appear as `NULL` inside the list, e.g. `[ev1, NULL, ev3]`.
- **`F.filter(array_col, lambda item: item.isNotNull())`**: Keeps only non-null items in the list, e.g. `[ev1, NULL, ev3]` -> `[ev1, ev3]`.
- **`F.transform(array_col, lambda item: item["rule_id"])`**: Extracts one field from each struct in the list, e.g. `[ev1, ev3]` -> `["RULE_01_CARD_VELOCITY_1H", "RULE_03_REPEATED_SIMILAR_MERCHANT_1H"]`.

In [ ]:
# Step 3d: Combine matched rule evidence into one array per transaction and display flagged alerts
filtered_evidence = F.filter(
    F.array(rule_01_evidence, rule_02_evidence, rule_03_evidence),
    lambda item: item.isNotNull(),
)

evaluated_df = (
    flagged_df.withColumn("ruleset_version", F.lit(ACTIVE_RULESET_VERSION))
    .withColumn("rule_evidence", filtered_evidence)
    .withColumn(
        "matched_rule_ids",
        F.transform(F.col("rule_evidence"), lambda item: item["rule_id"]),
    )
    .withColumn(
        "matched_rule_versions",
        F.transform(F.col("rule_evidence"), lambda item: item["rule_version"]),
    )
    .withColumn("matched_rule_count", F.size(F.col("matched_rule_ids")))
    .withColumn("is_flagged", F.col("matched_rule_count") > 0)
)

eval_counts = evaluated_df.agg(
    F.count("*").alias("evaluated_rows"),
    F.countDistinct("trans_num").alias("distinct_trans_num"),
    F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("flagged_transactions"),
).collect()[0]

print(
    f"Evaluated {eval_counts['evaluated_rows']:,} transactions "
    f"({eval_counts['distinct_trans_num']:,} unique IDs) -> "
    f"{eval_counts['flagged_transactions']:,} flagged alerts."
)
display(
    evaluated_df.filter(F.col("is_flagged"))
    .select(
        "trans_num",
        "cc_num",
        "merchant",
        "amount",
        "event_time",
        *FEATURE_COLUMNS,
        "unusual_amount_threshold_14d",
        "matched_rule_ids",
        "matched_rule_versions",
        "rule_evidence",
    )
    .orderBy("event_time", "trans_num")
)

### Step 4: Save Flagged Alerts to Gold (`workspace.sentinel_gold.alerts`)

We filter `evaluated_df` to `is_flagged == True`, derive a deterministic `alert_id = sha2(ruleset_version || '|' || trans_num, 256)`, and overwrite `workspace.sentinel_gold.alerts`.
- **Note on Spark laziness**: Referencing `evaluated_df` avoids repeating transformation code, though Spark DataFrames are lazy query plans and still execute the lineage on each action unless cached.

In [ ]:
# Step 4: Filter to flagged alerts, compute deterministic alert_id, and save to Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

gold_alerts_df = (
    evaluated_df.filter(F.col("is_flagged"))
    .withColumn(
        "alert_id",
        F.sha2(F.concat_ws("|", F.col("ruleset_version"), F.col("trans_num")), 256),
    )
    .select(*GOLD_ALERTS_COLUMNS)
)

gold_alerts_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(GOLD_ALERTS_TABLE)

print(f"Saved {eval_counts['flagged_transactions']:,} flagged alerts to {GOLD_ALERTS_TABLE}.")

### Step 5: Basic Gold Verification

We read `workspace.sentinel_gold.alerts` back from Delta storage and check row count, unique `alert_id` and `trans_num` counts, and confirm `rule_evidence` is never `NULL` or empty (`size < 1`).

In [ ]:
# Step 5: Read back workspace.sentinel_gold.alerts and check counts, unique IDs, and non-null evidence
saved_gold_alerts = spark.read.table(GOLD_ALERTS_TABLE)

gold_check = saved_gold_alerts.agg(
    F.count("*").alias("saved_rows"),
    F.countDistinct("alert_id").alias("distinct_alert_ids"),
    F.countDistinct("trans_num").alias("distinct_trans_nums"),
    F.sum(
        F.when(
            F.col("alert_id").isNull()
            | F.col("trans_num").isNull()
            | F.col("rule_evidence").isNull()
            | (F.size(F.col("rule_evidence")) < 1),
            1,
        ).otherwise(0)
    ).alias("missing_evidence_rows"),
).collect()[0]

expected_flagged = int(eval_counts["flagged_transactions"] or 0)
assert gold_check["saved_rows"] == expected_flagged, (
    f"Expected {expected_flagged} Gold rows, found {gold_check['saved_rows']}"
)
assert gold_check["distinct_alert_ids"] == expected_flagged, "Duplicate alert_id in Gold alerts"
assert gold_check["distinct_trans_nums"] == expected_flagged, "Duplicate trans_num in Gold alerts"
assert int(gold_check["missing_evidence_rows"] or 0) == 0, "NULL or empty rule_evidence in Gold alerts"

print(
    f"Verified {GOLD_ALERTS_TABLE}: {gold_check['saved_rows']:,} rows, "
    f"{gold_check['distinct_alert_ids']:,} unique alert_ids, "
    f"{gold_check['distinct_trans_nums']:,} unique trans_nums, "
    f"{int(gold_check['missing_evidence_rows'] or 0)} missing-evidence rows."
)
display(saved_gold_alerts.orderBy("event_time", "trans_num"))

In [ ]:
# Step 6: Rule-development summary for January 15-28, 2019 (after 14 days of warm-up history)
assert "is_fraud" not in evaluated_df.columns, "Fraud labels must not appear in evaluated data"

rule_dev_df = evaluated_df.filter(
    F.col("replay_batch").between(F.lit("2019-01-15"), F.lit("2019-01-28"))
)

rule_dev_summary = (
    rule_dev_df.agg(
        F.count("*").alias("evaluated_transactions"),
        F.sum(F.when(F.col("is_flagged"), 1).otherwise(0)).alias("flagged_transactions"),
        F.round(F.avg(F.when(F.col("is_flagged"), 100.0).otherwise(0.0)), 2).alias("flagged_pct"),
        F.sum(F.when(F.col("rule_01_velocity_match"), 1).otherwise(0)).alias("rule_01_velocity_1h_matches"),
        F.sum(F.when(F.col("rule_02_unusual_amount_match"), 1).otherwise(0)).alias("rule_02_unusual_amount_14d_matches"),
        F.sum(F.when(F.col("rule_03_similar_merchant_match"), 1).otherwise(0)).alias("rule_03_repeated_similar_merchant_1h_matches"),
        F.sum(F.when(F.col("matched_rule_count") > 1, 1).otherwise(0)).alias("multi_rule_transactions"),
    )
)

display(rule_dev_summary)